# Live schema-drift simulator

This Microsoft Fabric notebook sends small JSON batches to `RawTelemetry` for 30 minutes. Every event receives a new GUID and a UTC timestamp generated immediately before ingestion.

The stream begins with approved fields, then introduces `serviceCountdownHours`, an incompatible `signalStrength`, a nested `modem` object, and `compressorHealth`. This gives the dashboard time to show healthy traffic before drift appears.

> Run the repository KQL deployment files through `06-drift-log.kql` first. The notebook uses inline ingestion for a controlled demonstration, not as a production streaming architecture.

In [ ]:
%pip install -q azure-kusto-data

## Configure the run

Copy the Query URI from the Eventhouse or KQL database details. Keep the cleanup switch off initially so the completed run remains available for dashboard exploration and review.

In [ ]:
EVENTHOUSE_QUERY_URI = "https://<your-eventhouse-query-uri>"
KQL_DATABASE = "<your-kql-database-name>"

RUN_MINUTES = 30
BATCH_INTERVAL_SECONDS = 30
EVENTS_PER_SOURCE_PER_BATCH = 4

# Cleanup is deliberately opt-in. Change this only after reviewing the run.
CLEAN_UP = False

if "<" in EVENTHOUSE_QUERY_URI or "<" in KQL_DATABASE:
    raise ValueError("Set EVENTHOUSE_QUERY_URI and KQL_DATABASE before continuing.")

## Connect to Eventhouse

The notebook obtains the signed-in Fabric user's token. That identity needs permission to ingest into `RawTelemetry`, query the tables, and run delete commands if cleanup is enabled.

In [ ]:
import json
import random
import time
import uuid
from datetime import datetime, timezone

import notebookutils
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder
from azure.kusto.data.helpers import dataframe_from_result_table

access_token = notebookutils.credentials.getToken(EVENTHOUSE_QUERY_URI)
connection = KustoConnectionStringBuilder.with_aad_application_token_authentication(
    EVENTHOUSE_QUERY_URI, access_token
)
client = KustoClient(connection)

# One physical envelope value identifies every row produced by this run.
RUN_ID = uuid.uuid4().hex[:12]
RUN_TENANT_ID = f"tenant-simulator-{RUN_ID}"
print(f"Simulation run: {RUN_ID}")
print(f"Cleanup key: {RUN_TENANT_ID}")

## Generate synthetic events

The elapsed-time checks introduce drift gradually:

- minute 5: controller messages add `serviceCountdownHours`;
- minute 10: some gateway messages send `signalStrength` as `"unknown"`;
- minute 12: gateway messages add a nested `modem` object;
- minute 15: cooling-unit messages add `compressorHealth`.

In [ ]:
randomizer = random.Random()

def utc_now_text():
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds").replace("+00:00", "Z")

def envelope(source_type, asset_number, schema_version, telemetry):
    return {
        "id": str(uuid.uuid4()),
        "timestamp": utc_now_text(),
        "messageType": "telemetry",
        "sourceType": source_type,
        "tenantId": RUN_TENANT_ID,
        "assetId": f"sim-{source_type}-{asset_number:02d}",
        "deviceId": f"sim-device-{source_type}-{asset_number:02d}",
        "schemaVersion": schema_version,
        "simulationRunId": RUN_ID,
        "payload": {"version": "1.0", "telemetry": telemetry},
    }

def build_batch(elapsed_minutes):
    events = []
    for asset_number in range(1, EVENTS_PER_SOURCE_PER_BATCH + 1):
        controller = {
            "controllerStatus": "running",
            "engineHours": round(1250 + elapsed_minutes / 60 + asset_number, 2),
            "fuelConsumption": round(randomizer.uniform(7.8, 8.8), 2),
        }
        controller_version = 1
        if elapsed_minutes >= 5:
            controller["serviceCountdownHours"] = round(120 - elapsed_minutes, 1)
            controller_version = 2
        events.append(envelope("controller", asset_number, controller_version, controller))

        gateway = {
            "hdop": round(randomizer.uniform(0.6, 1.2), 2),
            "primaryVoltage": round(randomizer.uniform(12.3, 13.0), 2),
            "speed": round(randomizer.uniform(42, 62), 1),
            "fixType": "3d",
            "signalStrength": randomizer.randint(14, 24),
        }
        gateway_version = 1
        if elapsed_minutes >= 10 and asset_number == 1:
            gateway["signalStrength"] = "unknown"
            gateway_version = 2
        if elapsed_minutes >= 12:
            gateway["modem"] = {"firmware": "3.2.1"}
            gateway_version = 2
        events.append(envelope("gateway", asset_number, gateway_version, gateway))

        zone_count = 2 if asset_number % 2 else 1
        zones = [
            {
                "zone": zone_number,
                "isDoorOpen": randomizer.random() < 0.08,
                "isActive": True,
                "operatingMode": "cool",
                "returnAirTemperature": round(randomizer.uniform(2.0, 5.0), 1),
                "dischargeAirTemperature": round(randomizer.uniform(0.5, 2.0), 1),
                "setpointTemperature": 2.0 if zone_number == 1 else 4.0,
            }
            for zone_number in range(1, zone_count + 1)
        ]
        cooling_unit = {
            "controllerStatus": "running",
            "fuelLevelPercent": round(randomizer.uniform(55, 80), 1),
            "ambientTemperature": round(randomizer.uniform(25, 33), 1),
            "numberOfZones": zone_count,
            "zones": zones,
        }
        cooling_version = 1
        if elapsed_minutes >= 15:
            cooling_unit["compressorHealth"] = "warning" if asset_number == 1 else "healthy"
            cooling_version = 2
        events.append(envelope("cooling_unit", asset_number, cooling_version, cooling_unit))

    return events

## Run the 30-minute stream

Leave this cell running. It sends one small batch every 30 seconds and prints periodic progress. Stopping the cell ends future ingestion but does not remove rows already written.

In [ ]:
def ingest_json_batch(events):
    json_lines = "\n".join(json.dumps(event, separators=(",", ":")) for event in events)
    command = (
        ".ingest inline into table RawTelemetry "
        "with (format='json', ingestionMappingReference='RawTelemetryJsonMapping') <|\n"
        + json_lines
    )
    client.execute_mgmt(KQL_DATABASE, command)

started_monotonic = time.monotonic()
deadline = started_monotonic + RUN_MINUTES * 60
batch_number = 0
event_count = 0
print(f"Started at {utc_now_text()} and scheduled for {RUN_MINUTES} minutes.")

try:
    while time.monotonic() < deadline:
        elapsed_minutes = (time.monotonic() - started_monotonic) / 60
        batch = build_batch(elapsed_minutes)
        ingest_json_batch(batch)
        batch_number += 1
        event_count += len(batch)
        if batch_number == 1 or batch_number % 10 == 0:
            print(
                f"{utc_now_text()} | elapsed={elapsed_minutes:.1f}m "
                f"| batches={batch_number} | events={event_count}"
            )
        remaining_seconds = deadline - time.monotonic()
        if remaining_seconds > 0:
            time.sleep(min(BATCH_INTERVAL_SECONDS, remaining_seconds))
except KeyboardInterrupt:
    print("Simulation stopped from the notebook. Already-ingested rows remain available.")

print(f"Finished at {utc_now_text()} after sending {event_count} events in {batch_number} batches.")

## Verify this run

This summary is scoped to the unique simulator tenant. Update-policy results can arrive shortly after the raw batch, so rerun the cell if the counts are still settling.

In [ ]:
run_summary_query = f"""
let RunMessages = RawTelemetry
    | where TenantId == '{RUN_TENANT_ID}'
    | project MessageId;
union
    (RawTelemetry | where TenantId == '{RUN_TENANT_ID}' | summarize Rows=count() | extend Table='RawTelemetry'),
    (ControllerTelemetry | where TenantId == '{RUN_TENANT_ID}' | summarize Rows=count() | extend Table='ControllerTelemetry'),
    (GatewayTelemetry | where TenantId == '{RUN_TENANT_ID}' | summarize Rows=count() | extend Table='GatewayTelemetry'),
    (CoolingUnitTelemetry | where TenantId == '{RUN_TENANT_ID}' | summarize Rows=count() | extend Table='CoolingUnitTelemetry'),
    (CoolingUnitZones | where TenantId == '{RUN_TENANT_ID}' | summarize Rows=count() | extend Table='CoolingUnitZones'),
    (TelemetryDriftObservations | where MessageId in (RunMessages) | summarize Rows=count() | extend Table='TelemetryDriftObservations')
| project Table, Rows
| order by Table asc
"""
response = client.execute(KQL_DATABASE, run_summary_query)
display(dataframe_from_result_table(response.primary_results[0]))

## Keep or clean up

**Recommendation:** keep the run after the first demonstration. It supports dashboard exploration, alert validation, and promotion exercises without another 30-minute wait. Clean it only when the test database must return to its prior state or before a repeatable benchmark.

Cleanup is scoped to this run's unique tenant and deletes dependent rows before raw rows. Set `CLEAN_UP = True` in the configuration cell, rerun that cell, and then run the cleanup cell below. Kusto record deletion is an administrative operation and should be used only in the disposable test database.

In [ ]:
if not CLEAN_UP:
    print(f"Keeping simulation data for {RUN_TENANT_ID}. Set CLEAN_UP = True to remove it.")
else:
    cleanup_commands = [
        f""".delete table TelemetryDriftObservations records <|
TelemetryDriftObservations
| where MessageId in (RawTelemetry | where TenantId == '{RUN_TENANT_ID}' | project MessageId)""",
        f".delete table CoolingUnitZones records <| CoolingUnitZones | where TenantId == '{RUN_TENANT_ID}'",
        f".delete table ControllerTelemetry records <| ControllerTelemetry | where TenantId == '{RUN_TENANT_ID}'",
        f".delete table GatewayTelemetry records <| GatewayTelemetry | where TenantId == '{RUN_TENANT_ID}'",
        f".delete table CoolingUnitTelemetry records <| CoolingUnitTelemetry | where TenantId == '{RUN_TENANT_ID}'",
        f".delete table RawTelemetry records <| RawTelemetry | where TenantId == '{RUN_TENANT_ID}'",
    ]
    for cleanup_command in cleanup_commands:
        client.execute_mgmt(KQL_DATABASE, cleanup_command)
    print(f"Cleanup commands completed for {RUN_TENANT_ID}.")